In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import csv
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from collections import Counter, defaultdict
 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
 
# Verify
pt_files = [f for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')]
print(f"Features dir: {FEATURES_DIR}")
print(f".pt files found: {len(pt_files)}")
 
# === Original dataset (for CSVs) ===
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
ZIP1 = os.path.join(DATASET_DIR, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")
 
sample_csv_path = os.path.join(ZIP1, "sample.csv")
annotation_csv_path = os.path.join(ZIP1, "annotation.csv")
print(f"sample.csv exists: {os.path.exists(sample_csv_path)}")
print(f"annotation.csv exists: {os.path.exists(annotation_csv_path)}")
 
# %%
# Load annotations
annotations = {}
with open(annotation_csv_path, 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            clip_id = row[0].strip()
            annotations[clip_id] = {
                'text': row[1].strip() if len(row) > 1 else '',
                'emotion': row[7].strip() if len(row) > 7 and row[7].strip() else None,
                'polarity': row[5].strip() if len(row) > 5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row) > 6 and row[6].strip() else None,
                'uncertainty': row[8].strip() if len(row) > 8 and row[8].strip() else None,
            }
 
# Load samples and build index
samples = []
with open(sample_csv_path, 'r') as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        samples.append(row)
 
emotion_map = {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 
               'neutral': 4, 'sad': 5, 'surprise': 6}
polarity_map = {'positive': 0, 'neutral': 1, 'negative': 2}
intensity_map = {'weak': 0, 'powerful': 1}
emo_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
 
mcis_index = []
for s in samples:
    sample_id = s[0].strip()
    clips = [s[i].strip() for i in range(1, 5)]
    entry = {
        'sample_id': sample_id,
        'clip_ids': clips,
        'feature_files': [c.replace('/', '_') + '.pt' for c in clips],
    }
    for label_name, clip_idx in [('clip3', 2), ('clip4', 3)]:
        cid = clips[clip_idx]
        if cid in annotations and annotations[cid].get('emotion'):
            ann = annotations[cid]
            entry[f'{label_name}_emotion'] = emotion_map.get(ann['emotion'], -1)
            entry[f'{label_name}_emotion_str'] = ann['emotion']
            entry[f'{label_name}_polarity'] = polarity_map.get(ann.get('polarity', ''), -1)
            entry[f'{label_name}_intensity'] = intensity_map.get(ann.get('intensity', ''), -1)
            entry[f'{label_name}_uncertainty'] = int(ann['uncertainty']) if ann.get('uncertainty', '').isdigit() else -1
        else:
            entry[f'{label_name}_emotion'] = -1
            entry[f'{label_name}_emotion_str'] = None
            entry[f'{label_name}_polarity'] = -1
            entry[f'{label_name}_intensity'] = -1
            entry[f'{label_name}_uncertainty'] = -1
    mcis_index.append(entry)
 
print(f"\nTotal MCIS samples: {len(mcis_index)}")
print(f"\nClip IV (target) emotion distribution:")
for emo_id, count in sorted(Counter(e['clip4_emotion'] for e in mcis_index).items()):
    if emo_id >= 0:
        print(f"  {emo_names[emo_id]}: {count} ({100*count/len(mcis_index):.1f}%)")

Features dir: /kaggle/input/datasets/ptrnghieu/hi-ef-features-v2
.pt files found: 7925
sample.csv exists: True
annotation.csv exists: True

Total MCIS samples: 2830

Clip IV (target) emotion distribution:
  angry: 525 (18.6%)
  disgust: 260 (9.2%)
  fear: 38 (1.3%)
  happy: 662 (23.4%)
  neutral: 652 (23.0%)
  sad: 419 (14.8%)
  surprise: 274 (9.7%)


In [4]:
def split_mcis(mcis_index, train_ratio=0.7, val_ratio=0.15, seed=42):
    """Split MCIS into train/val/test, avoiding clip4 leakage."""
    rng = random.Random(seed)
    
    clip4_groups = {}
    for idx, entry in enumerate(mcis_index):
        c4 = entry['clip_ids'][3]
        if c4 not in clip4_groups:
            clip4_groups[c4] = []
        clip4_groups[c4].append(idx)
    
    group_keys = list(clip4_groups.keys())
    rng.shuffle(group_keys)
    
    n_groups = len(group_keys)
    n_train = int(n_groups * train_ratio)
    n_val = int(n_groups * val_ratio)
    
    train_idx = [i for g in group_keys[:n_train] for i in clip4_groups[g]]
    val_idx = [i for g in group_keys[n_train:n_train+n_val] for i in clip4_groups[g]]
    test_idx = [i for g in group_keys[n_train+n_val:] for i in clip4_groups[g]]
    
    return train_idx, val_idx, test_idx
 
train_idx, val_idx, test_idx = split_mcis(mcis_index)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
 
# Verify no clip4 overlap
train_c4 = set(mcis_index[i]['clip_ids'][3] for i in train_idx)
val_c4 = set(mcis_index[i]['clip_ids'][3] for i in val_idx)
test_c4 = set(mcis_index[i]['clip_ids'][3] for i in test_idx)
assert len(train_c4 & test_c4) == 0, "Clip4 leakage between train and test!"
print("No clip4 leakage ✓")

Train: 1980, Val: 424, Test: 426
No clip4 leakage ✓


In [5]:
class HiEFDataset(Dataset):
    def __init__(self, mcis_index, features_dir, indices):
        self.mcis_index = mcis_index
        self.features_dir = features_dir
        self.indices = indices
    
    def __len__(self):
        return len(self.indices)
    
    def _load_clip(self, feature_file):
        path = os.path.join(self.features_dir, feature_file)
        data = torch.load(path, map_location='cpu', weights_only=False)
        return {
            'face': data['face_features'],                         # [16, 512]
            'ori': data['ori_features'],                           # [16, 512]
            'text': data['text_feature'],                          # [512]
            'audio': data.get('audio_feature', torch.zeros(527)),  # [527]
        }
    
    def __getitem__(self, idx):
        entry = self.mcis_index[self.indices[idx]]
        
        clip1 = self._load_clip(entry['feature_files'][0])
        clip2 = self._load_clip(entry['feature_files'][1])
        clip3 = self._load_clip(entry['feature_files'][2])
        
        return {
            'clip1_face': clip1['face'], 'clip1_ori': clip1['ori'],
            'clip1_text': clip1['text'], 'clip1_audio': clip1['audio'],
            'clip2_face': clip2['face'], 'clip2_ori': clip2['ori'],
            'clip2_text': clip2['text'], 'clip2_audio': clip2['audio'],
            'clip3_face': clip3['face'], 'clip3_ori': clip3['ori'],
            'clip3_text': clip3['text'], 'clip3_audio': clip3['audio'],
            'target': entry['clip4_emotion'],
            'clip3_emotion': entry['clip3_emotion'],
            'clip4_polarity': entry['clip4_polarity'],
            'clip4_intensity': entry['clip4_intensity'],
            'clip4_uncertainty': entry['clip4_uncertainty'],
        }
 
 
def collate_fn(batch):
    result = {}
    tensor_keys = [f'clip{c}_{m}' for c in [1,2,3] for m in ['face','ori','text','audio']]
    for key in tensor_keys:
        result[key] = torch.stack([b[key] for b in batch])
    
    for key in ['target', 'clip3_emotion', 'clip4_polarity', 'clip4_intensity', 'clip4_uncertainty']:
        result[key] = torch.tensor([b[key] for b in batch], dtype=torch.long)
    
    return result
 
# %%
BATCH_SIZE = 32
 
train_dataset = HiEFDataset(mcis_index, FEATURES_DIR, train_idx)
val_dataset = HiEFDataset(mcis_index, FEATURES_DIR, val_idx)
test_dataset = HiEFDataset(mcis_index, FEATURES_DIR, test_idx)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)
 
# Verify shapes
batch = next(iter(train_loader))
print("Batch shapes:")
for k, v in batch.items():
    print(f"  {k}: {v.shape}")

Batch shapes:
  clip1_face: torch.Size([32, 16, 512])
  clip1_ori: torch.Size([32, 16, 512])
  clip1_text: torch.Size([32, 512])
  clip1_audio: torch.Size([32, 527])
  clip2_face: torch.Size([32, 16, 512])
  clip2_ori: torch.Size([32, 16, 512])
  clip2_text: torch.Size([32, 512])
  clip2_audio: torch.Size([32, 527])
  clip3_face: torch.Size([32, 16, 512])
  clip3_ori: torch.Size([32, 16, 512])
  clip3_text: torch.Size([32, 512])
  clip3_audio: torch.Size([32, 527])
  target: torch.Size([32])
  clip3_emotion: torch.Size([32])
  clip4_polarity: torch.Size([32])
  clip4_intensity: torch.Size([32])
  clip4_uncertainty: torch.Size([32])


In [6]:
class TemporalTransformer(nn.Module):
    """Self-attention over frame-level features: [B, T, D] → [B, T, D]"""
    
    def __init__(self, d_model=512, n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, 16, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
    
    def forward(self, x):
        x = x + self.pos_encoding[:, :x.size(1), :]
        return self.transformer(x)
 
 
class CrossAttentionFusion(nn.Module):
    """Query attends to key-value inputs: [B,Q,D] x [B,K,D] → [B,Q,D]"""
    
    def __init__(self, d_model=512, n_heads=8, n_layers=1, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
    
    def forward(self, query, key_values):
        x = query
        for attn, norm in zip(self.layers, self.norms):
            attended, _ = attn(x, key_values, key_values)
            x = norm(x + attended)
        return x
 
 
class IntraVideoFusion(nn.Module):
    """Process one clip: face + ori + text + audio → single 512-d feature."""
    
    def __init__(self, d_model=512, audio_dim=527):
        super().__init__()
        self.face_temporal = TemporalTransformer(d_model, n_heads=8, n_layers=2)
        self.ori_temporal = TemporalTransformer(d_model, n_heads=8, n_layers=2)
        self.type_fusion = CrossAttentionFusion(d_model, n_heads=8, n_layers=1)
        self.audio_proj = nn.Linear(audio_dim, d_model)
        self.modality_fusion = CrossAttentionFusion(d_model, n_heads=8, n_layers=1)
    
    def forward(self, face_features, ori_features, text_feature, audio_feature):
        """
        face_features: [B, 16, 512]
        ori_features:  [B, 16, 512]
        text_feature:  [B, 512]
        audio_feature: [B, 527]
        """
        # Temporal modeling
        face_out = self.face_temporal(face_features).mean(dim=1, keepdim=True)  # [B, 1, 512]
        ori_out = self.ori_temporal(ori_features).mean(dim=1, keepdim=True)     # [B, 1, 512]
        
        # Type fusion: face attends to [face, ori]
        vis_stack = torch.cat([face_out, ori_out], dim=1)         # [B, 2, 512]
        video_feat = self.type_fusion(face_out, vis_stack)         # [B, 1, 512]
        
        # Audio: normalize then project 527 → 512
        audio_normed = F.normalize(audio_feature, dim=-1)
        audio_feat = self.audio_proj(audio_normed).unsqueeze(1)    # [B, 1, 512]
        
        # Text
        text_feat = text_feature.unsqueeze(1)                      # [B, 1, 512]
        
        # Modality fusion: video attends to [video, text, audio]
        mod_stack = torch.cat([video_feat, text_feat, audio_feat], dim=1)  # [B, 3, 512]
        clip_feat = self.modality_fusion(video_feat, mod_stack)            # [B, 1, 512]
        
        return clip_feat.squeeze(1)  # [B, 512]
 
 
class InterVideoFusion(nn.Module):
    """Fuse clips I, II, III via LSTM + Transformer → 512-d feature."""
    
    def __init__(self, d_model=512, lstm_layers=3, transformer_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=d_model, hidden_size=d_model,
                            num_layers=lstm_layers, batch_first=False, dropout=0.1)
        self.pos_encoding = nn.Parameter(torch.randn(1, 3, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)
    
    def forward(self, clip1_feat, clip2_feat, clip3_feat):
        # LSTM: temporal trajectory [3, B, 512]
        seq = torch.stack([clip1_feat, clip2_feat, clip3_feat], dim=0)
        lstm_out, _ = self.lstm(seq)
        
        # Transformer: attend to most relevant [B, 3, 512]
        trans_in = lstm_out.permute(1, 0, 2) + self.pos_encoding
        trans_out = self.transformer(trans_in)
        
        return trans_out.mean(dim=1)  # [B, 512]
 
 
class EFBaseline(nn.Module):
    """Full Emotion Forecasting baseline.
    
    Shared intra-video fusion for all clips (baseline behavior).
    LSTM+Transformer inter-video fusion.
    """
    
    def __init__(self, d_model=512, n_classes=7):
        super().__init__()
        self.intra_fusion = IntraVideoFusion(d_model)
        self.inter_fusion = InterVideoFusion(d_model)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.3),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(d_model // 2, n_classes),
        )
    
    def forward(self, batch):
        clip1 = self.intra_fusion(batch['clip1_face'], batch['clip1_ori'], 
                                   batch['clip1_text'], batch['clip1_audio'])
        clip2 = self.intra_fusion(batch['clip2_face'], batch['clip2_ori'], 
                                   batch['clip2_text'], batch['clip2_audio'])
        clip3 = self.intra_fusion(batch['clip3_face'], batch['clip3_ori'], 
                                   batch['clip3_text'], batch['clip3_audio'])
        
        final = self.inter_fusion(clip1, clip2, clip3)
        return self.classifier(final)
 
# %%
model = EFBaseline(d_model=512, n_classes=7).to(DEVICE)
 
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
 
# Verify forward pass
with torch.no_grad():
    batch_gpu = {k: v.to(DEVICE) for k, v in batch.items()}
    logits = model(batch_gpu)
    print(f"Forward pass OK: {logits.shape}")

/tmp/ipykernel_58/2553630346.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
/tmp/ipykernel_58/2553630346.py:89: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)


Total parameters:     27,743,751
Trainable parameters: 27,743,751
Forward pass OK: torch.Size([32, 7])


In [7]:
def compute_metrics(preds, labels, n_classes=7):
    preds, labels = np.array(preds), np.array(labels)
    war = (preds == labels).sum() / len(labels) * 100
    recalls = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0:
            recalls.append((preds[mask] == c).sum() / mask.sum() * 100)
    uar = np.mean(recalls) if recalls else 0.0
    return war, uar
 
 
def run_epoch(model, loader, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    
    total_loss = 0
    all_preds, all_labels = [], []
    
    ctx = torch.enable_grad if is_train else torch.no_grad
    with ctx():
        for batch in loader:
            batch_gpu = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch_gpu)
            loss = F.cross_entropy(logits, batch_gpu['target'])
            
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            
            total_loss += loss.item() * logits.size(0)
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(batch['target'].numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    war, uar = compute_metrics(all_preds, all_labels)
    return avg_loss, war, uar, all_preds, all_labels
 
# %%
N_EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-5
 
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)
 
best_val_uar = 0
best_epoch = 0
history = defaultdict(list)
 
print(f"{'Ep':>3} | {'Tr Loss':>7} | {'Tr WAR':>6} | {'Tr UAR':>6} | "
      f"{'Va Loss':>7} | {'Va WAR':>6} | {'Va UAR':>6} | {'LR':>8}")
print("-" * 75)
 
for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_war, tr_uar, _, _ = run_epoch(model, train_loader, optimizer)
    va_loss, va_war, va_uar, _, _ = run_epoch(model, val_loader)
    
    scheduler.step(va_loss)
    lr = optimizer.param_groups[0]['lr']
    
    for k, v in [('tr_loss',tr_loss),('va_loss',va_loss),('tr_war',tr_war),
                  ('va_war',va_war),('tr_uar',tr_uar),('va_uar',va_uar)]:
        history[k].append(v)
    
    star = ''
    if va_uar > best_val_uar:
        best_val_uar = va_uar
        best_epoch = epoch
        torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
        star = ' ★'
    
    print(f"{epoch:>3} | {tr_loss:>7.4f} | {tr_war:>5.1f}% | {tr_uar:>5.1f}% | "
          f"{va_loss:>7.4f} | {va_war:>5.1f}% | {va_uar:>5.1f}% | {lr:>8.6f}{star}")
 
print(f"\nBest val UAR: {best_val_uar:.2f}% at epoch {best_epoch}")

 Ep | Tr Loss | Tr WAR | Tr UAR | Va Loss | Va WAR | Va UAR |       LR
---------------------------------------------------------------------------
  1 |  1.8302 |  21.8% |  14.2% |  1.7686 |  24.1% |  14.3% | 0.000100 ★
  2 |  1.8034 |  25.0% |  16.0% |  1.7608 |  25.2% |  15.6% | 0.000100 ★
  3 |  1.7720 |  27.7% |  18.0% |  1.7331 |  29.2% |  17.6% | 0.000100 ★
  4 |  1.7139 |  33.8% |  21.9% |  1.7352 |  32.3% |  21.6% | 0.000100 ★
  5 |  1.6933 |  34.6% |  23.0% |  1.7022 |  35.1% |  22.2% | 0.000100 ★
  6 |  1.6680 |  36.9% |  24.4% |  1.6911 |  35.6% |  23.1% | 0.000100 ★
  7 |  1.6614 |  36.4% |  24.0% |  1.6868 |  35.1% |  23.1% | 0.000100 ★
  8 |  1.6497 |  38.7% |  25.6% |  1.6584 |  35.6% |  22.7% | 0.000100
  9 |  1.6013 |  42.0% |  28.4% |  1.7031 |  36.3% |  23.8% | 0.000100 ★
 10 |  1.5989 |  41.9% |  28.3% |  1.6387 |  36.1% |  23.7% | 0.000100
 11 |  1.5813 |  42.2% |  28.4% |  1.6799 |  38.4% |  25.0% | 0.000100 ★
 12 |  1.5457 |  43.3% |  29.0% |  1.6671 |  36.1% |  

In [8]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pt', map_location=DEVICE, weights_only=True))
te_loss, te_war, te_uar, te_preds, te_labels = run_epoch(model, test_loader)
 
print(f"=== Test Results ===")
print(f"WAR: {te_war:.2f}%")
print(f"UAR: {te_uar:.2f}%")
print(f"(Paper: WAR=35.19%, UAR=23.72%)")
 
te_preds, te_labels = np.array(te_preds), np.array(te_labels)
print(f"\nPer-class recall:")
for c in range(7):
    mask = te_labels == c
    if mask.sum() > 0:
        recall = (te_preds[mask] == c).sum() / mask.sum() * 100
        print(f"  {emo_names[c]:>10}: {recall:5.1f}% ({mask.sum()} samples)")

=== Test Results ===
WAR: 37.09%
UAR: 26.66%
(Paper: WAR=35.19%, UAR=23.72%)

Per-class recall:
       angry:  39.5% (76 samples)
     disgust:  20.0% (45 samples)
        fear:   0.0% (4 samples)
       happy:  40.4% (99 samples)
     neutral:  57.4% (108 samples)
         sad:  29.3% (58 samples)
    surprise:   0.0% (36 samples)


In [9]:
conf = np.zeros((7, 7), dtype=int)
for p, l in zip(te_preds, te_labels):
    conf[l][p] += 1
 
print("Confusion Matrix (rows=true, cols=predicted):")
header = f"{'':>10}" + "".join(f"{emo_names[i][:7]:>8}" for i in range(7))
print(header)
for i in range(7):
    row = f"{emo_names[i]:>10}"
    for j in range(7):
        row += f"{conf[i][j]:>8}"
    total = conf[i].sum()
    if total > 0:
        row += f"  ({conf[i][i]/total*100:.1f}%)"
    print(row)

Confusion Matrix (rows=true, cols=predicted):
             angry disgust    fear   happy neutral     sad surpris
     angry      30       5       0       9      17      15       0  (39.5%)
   disgust      16       9       0      10       3       7       0  (20.0%)
      fear       1       0       0       1       1       1       0  (0.0%)
     happy      17       6       0      40      24      12       0  (40.4%)
   neutral      21       4       0      15      62       6       0  (57.4%)
       sad      21       1       0      14       5      17       0  (29.3%)
  surprise      10       2       0      10       8       6       0  (0.0%)


In [10]:
train_trans = np.zeros((7, 7))
for idx in train_idx:
    e = mcis_index[idx]
    a, b = e['clip3_emotion'], e['clip4_emotion']
    if a >= 0 and b >= 0:
        train_trans[a][b] += 1
 
row_sums = train_trans.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
train_trans_prob = train_trans / row_sums
 
print("Empirical A→B transitions (training data):")
header = f"{'A \\ B':>10}" + "".join(f"{emo_names[i][:7]:>8}" for i in range(7))
print(header)
for i in range(7):
    row = f"{emo_names[i]:>10}"
    for j in range(7):
        row += f"{train_trans_prob[i][j]*100:>7.1f}%"
    row += f"  (n={int(train_trans[i].sum())})"
    print(row)
 
# Model's predicted transitions on test set
print(f"\nModel's predicted transitions (test set):")
pred_trans = np.zeros((7, 7))
for i, idx in enumerate(test_idx):
    a_emo = mcis_index[idx]['clip3_emotion']
    if a_emo >= 0:
        pred_trans[a_emo][te_preds[i]] += 1
 
pred_sums = pred_trans.sum(axis=1, keepdims=True)
pred_sums[pred_sums == 0] = 1
pred_trans_prob = pred_trans / pred_sums
 
print(header)
for i in range(7):
    row = f"{emo_names[i]:>10}"
    for j in range(7):
        row += f"{pred_trans_prob[i][j]*100:>7.1f}%"
    row += f"  (n={int(pred_trans[i].sum())})"
    print(row)

Empirical A→B transitions (training data):
     A \ B   angry disgust    fear   happy neutral     sad surpris
     angry   34.5%   12.5%    2.4%   12.3%   11.8%   17.0%    9.5%  (n=423)
   disgust   27.8%   15.4%    1.2%   22.8%   16.7%    8.0%    8.0%  (n=162)
      fear   24.0%   12.0%    4.0%   16.0%   12.0%   16.0%   16.0%  (n=25)
     happy   12.0%    7.3%    1.6%   46.3%   15.2%   10.4%    7.3%  (n=441)
   neutral   10.6%    7.8%    0.9%   15.5%   44.6%    7.3%   13.3%  (n=451)
       sad   15.2%    7.4%    1.1%   19.6%   14.1%   33.3%    9.3%  (n=270)
  surprise   15.9%    9.1%    0.5%   19.7%   28.8%   16.3%    9.6%  (n=208)

Model's predicted transitions (test set):
     A \ B   angry disgust    fear   happy neutral     sad surpris
     angry   45.1%    4.4%    0.0%   14.3%   18.7%   17.6%    0.0%  (n=91)
   disgust   27.5%   20.0%    0.0%   22.5%   20.0%   10.0%    0.0%  (n=40)
      fear   50.0%    0.0%    0.0%   12.5%   12.5%   25.0%    0.0%  (n=8)
     happy   14.1%    7.1

In [11]:
# %% [markdown]
# ---
# ## 11. Analysis 1: Per-Class Error Diagnosis

# %%
# Get full predictions with probabilities
model.load_state_dict(torch.load('/kaggle/working/best_model.pt', map_location=DEVICE, weights_only=True))
model.eval()

all_probs, all_preds, all_labels, all_meta = [], [], [], []
with torch.no_grad():
    for batch in test_loader:
        bg = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(bg)
        all_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
        all_preds.append(logits.argmax(-1).cpu().numpy())
        all_labels.append(batch['target'].numpy())
        all_meta.append({
            'clip3_emotion': batch['clip3_emotion'].numpy(),
            'clip4_polarity': batch['clip4_polarity'].numpy(),
            'clip4_intensity': batch['clip4_intensity'].numpy(),
            'clip4_uncertainty': batch['clip4_uncertainty'].numpy(),
        })

probs = np.concatenate(all_probs)
preds = np.concatenate(all_preds)
labels = np.concatenate(all_labels)
meta = {k: np.concatenate([m[k] for m in all_meta]) for k in all_meta[0].keys()}

# %%
for c in range(7):
    mask = labels == c
    n = mask.sum()
    if n == 0: continue
    recall = (preds[mask] == c).sum() / n * 100
    avg_true_prob = probs[mask, c].mean() * 100
    avg_conf = probs[mask].max(axis=1).mean() * 100
    
    print(f"\n{'='*50}")
    print(f"{emo_names[c].upper()} (n={n}, recall={recall:.1f}%)")
    print(f"  Avg prob for true class: {avg_true_prob:.1f}%")
    print(f"  Avg confidence: {avg_conf:.1f}%")
    
    wrong = preds[mask & (preds != c)]
    if len(wrong) > 0:
        print(f"  Misclassified as:")
        for pc, cnt in sorted(Counter(wrong).items(), key=lambda x: -x[1]):
            print(f"    → {emo_names[pc]:>10}: {cnt} ({100*cnt/n:.1f}%)")
    
    print(f"  By Party A's emotion:")
    for a in range(7):
        sub = mask & (meta['clip3_emotion'] == a)
        if sub.sum() >= 3:
            acc = (preds[sub] == c).sum() / sub.sum() * 100
            print(f"    A={emo_names[a]:>10}: {acc:5.1f}% ({sub.sum()} samples)")

# %%
# Why are fear/surprise never predicted?
print("\nAvg predicted probability per class (all test samples):")
avg_p = probs.mean(axis=0) * 100
for c in range(7):
    print(f"  {emo_names[c]:>10}: {avg_p[c]:5.2f}%  {'█' * int(avg_p[c] * 2)}")

print("\nClassifier bias values:")
bias = model.classifier[-1].bias.detach().cpu().numpy()
for c in range(7):
    print(f"  {emo_names[c]:>10}: {bias[c]:+.4f}")

print("\nTraining label distribution:")
train_dist = Counter(mcis_index[i]['clip4_emotion'] for i in train_idx)
for c in range(7):
    cnt = train_dist.get(c, 0)
    print(f"  {emo_names[c]:>10}: {cnt:>4} ({100*cnt/len(train_idx):5.1f}%)  {'█' * int(100*cnt/len(train_idx))}")


ANGRY (n=76, recall=39.5%)
  Avg prob for true class: 27.9%
  Avg confidence: 47.5%
  Misclassified as:
    →    neutral: 17 (22.4%)
    →        sad: 15 (19.7%)
    →      happy: 9 (11.8%)
    →    disgust: 5 (6.6%)
  By Party A's emotion:
    A=     angry:  44.8% (29 samples)
    A=   disgust:  38.5% (13 samples)
    A=     happy:  42.9% (7 samples)
    A=   neutral:  25.0% (8 samples)
    A=       sad:  40.0% (10 samples)
    A=  surprise:  28.6% (7 samples)

DISGUST (n=45, recall=20.0%)
  Avg prob for true class: 16.6%
  Avg confidence: 42.2%
  Misclassified as:
    →      angry: 16 (35.6%)
    →      happy: 10 (22.2%)
    →        sad: 7 (15.6%)
    →    neutral: 3 (6.7%)
  By Party A's emotion:
    A=     angry:  12.5% (8 samples)
    A=   disgust:  44.4% (9 samples)
    A=     happy:  25.0% (8 samples)
    A=   neutral:  33.3% (3 samples)
    A=       sad:   9.1% (11 samples)
    A=  surprise:   0.0% (6 samples)

FEAR (n=4, recall=0.0%)
  Avg prob for true class: 1.3%
  Avg con

In [12]:
# %% [markdown]
# ---
# ## 12. Analysis 2: Uncertainty × Polarity × Intensity vs Accuracy

# %%
unc_names = {1:'Certain', 2:'Borderline', 3:'Uncertain'}
pol_names = {0:'Positive', 1:'Neutral', 2:'Negative'}
int_names = {0:'Weak', 1:'Powerful'}

print("--- By Uncertainty ---")
for u in [1, 2, 3]:
    m = meta['clip4_uncertainty'] == u
    if m.sum() == 0: continue
    w, ua = compute_metrics(preds[m], labels[m])
    print(f"  {unc_names[u]:>12} (n={m.sum():>3}): WAR={w:.1f}%, UAR={ua:.1f}%")

print("\n--- By Polarity ---")
for p in [0, 1, 2]:
    m = meta['clip4_polarity'] == p
    if m.sum() == 0: continue
    w, ua = compute_metrics(preds[m], labels[m])
    print(f"  {pol_names[p]:>12} (n={m.sum():>3}): WAR={w:.1f}%, UAR={ua:.1f}%")

print("\n--- By Intensity ---")
for i in [0, 1]:
    m = meta['clip4_intensity'] == i
    if m.sum() == 0: continue
    w, ua = compute_metrics(preds[m], labels[m])
    print(f"  {int_names[i]:>12} (n={m.sum():>3}): WAR={w:.1f}%, UAR={ua:.1f}%")

print("\n--- Polarity × Intensity ---")
for p in [0, 1, 2]:
    for i in [0, 1]:
        m = (meta['clip4_polarity'] == p) & (meta['clip4_intensity'] == i)
        if m.sum() < 5: continue
        w, ua = compute_metrics(preds[m], labels[m])
        print(f"  {pol_names[p]:>8} + {int_names[i]:<8} (n={m.sum():>3}): WAR={w:.1f}%, UAR={ua:.1f}%")

--- By Uncertainty ---
       Certain (n=320): WAR=38.4%, UAR=27.3%
    Borderline (n= 57): WAR=38.6%, UAR=29.2%
     Uncertain (n= 49): WAR=26.5%, UAR=18.8%

--- By Polarity ---
      Positive (n=109): WAR=38.5%, UAR=39.4%
       Neutral (n=119): WAR=49.6%, UAR=20.1%
      Negative (n=198): WAR=28.8%, UAR=22.2%

--- By Intensity ---
          Weak (n=294): WAR=37.1%, UAR=27.0%
      Powerful (n=132): WAR=37.1%, UAR=37.3%

--- Polarity × Intensity ---
  Positive + Weak     (n= 68): WAR=39.7%, UAR=41.1%
  Positive + Powerful (n= 41): WAR=36.6%, UAR=37.6%
   Neutral + Weak     (n= 85): WAR=48.2%, UAR=19.5%
   Neutral + Powerful (n= 34): WAR=52.9%, UAR=36.0%
  Negative + Weak     (n=141): WAR=29.1%, UAR=21.5%
  Negative + Powerful (n= 57): WAR=28.1%, UAR=40.7%


In [13]:
# %% [markdown]
# ---
# ## 13. Analysis 3: Ablation Studies
# 
# Retrains smaller models with different modality/clip configs.
# ~20-30 min total.

# %%
# Ablation model with configurable modalities and clips
class EFAblation(nn.Module):
    def __init__(self, d_model=512, n_classes=7, use_text=True, use_audio=True, clip_ids=[0,1,2]):
        super().__init__()
        self.clip_ids = clip_ids
        self.use_text = use_text
        self.use_audio = use_audio
        
        self.face_temp = TemporalTransformer(d_model)
        self.ori_temp = TemporalTransformer(d_model)
        self.type_fusion = CrossAttentionFusion(d_model)
        self.audio_proj = nn.Linear(527, d_model) if use_audio else None
        self.mod_fusion = CrossAttentionFusion(d_model)
        
        if len(clip_ids) > 1:
            self.lstm = nn.LSTM(d_model, d_model, num_layers=3, batch_first=False, dropout=0.1)
            self.inter_pos = nn.Parameter(torch.randn(1, len(clip_ids), d_model) * 0.02)
            el = nn.TransformerEncoderLayer(d_model, 8, d_model*4, 0.1, batch_first=True, norm_first=True)
            self.inter_trans = nn.TransformerEncoder(el, num_layers=2)
        else:
            self.lstm = None
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(0.3),
            nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(d_model//2, n_classes))
    
    def _intra(self, face, ori, text, audio):
        f = self.face_temp(face).mean(1, keepdim=True)
        o = self.ori_temp(ori).mean(1, keepdim=True)
        v = self.type_fusion(f, torch.cat([f, o], 1))
        mods = [v]
        if self.use_text: mods.append(text.unsqueeze(1))
        if self.use_audio and self.audio_proj:
            mods.append(self.audio_proj(F.normalize(audio, dim=-1)).unsqueeze(1))
        return self.mod_fusion(v, torch.cat(mods, 1)).squeeze(1)
    
    def forward(self, batch):
        feats = []
        for cid in self.clip_ids:
            p = f'clip{cid+1}'
            feats.append(self._intra(batch[f'{p}_face'], batch[f'{p}_ori'],
                                      batch[f'{p}_text'], batch[f'{p}_audio']))
        if self.lstm and len(feats) > 1:
            seq = torch.stack(feats, 0)
            out, _ = self.lstm(seq)
            out = out.permute(1,0,2) + self.inter_pos[:,:len(feats),:]
            final = self.inter_trans(out).mean(1)
        else:
            final = feats[0]
        return self.classifier(final)

def quick_train(model, train_loader, val_loader, test_loader, n_epochs=30, lr=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    best_uar = 0
    for ep in range(n_epochs):
        model.train()
        for b in train_loader:
            bg = {k:v.to(DEVICE) for k,v in b.items()}
            loss = F.cross_entropy(model(bg), bg['target'])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vp,vl = [],[]
        vl_loss = 0
        with torch.no_grad():
            for b in val_loader:
                bg = {k:v.to(DEVICE) for k,v in b.items()}
                lo = model(bg)
                vl_loss += F.cross_entropy(lo, bg['target']).item() * lo.size(0)
                vp.extend(lo.argmax(-1).cpu().numpy()); vl.extend(b['target'].numpy())
        sch.step(vl_loss / len(val_loader.dataset))
        _, vu = compute_metrics(vp, vl)
        if vu > best_uar:
            best_uar = vu
            torch.save(model.state_dict(), '/kaggle/working/abl.pt')
    
    model.load_state_dict(torch.load('/kaggle/working/abl.pt', map_location=DEVICE, weights_only=True))
    model.eval(); tp,tl=[],[]
    with torch.no_grad():
        for b in test_loader:
            bg = {k:v.to(DEVICE) for k,v in b.items()}
            tp.extend(model(bg).argmax(-1).cpu().numpy()); tl.extend(b['target'].numpy())
    tw, tu = compute_metrics(tp, tl)
    return tw, tu

# %%
# 3a: Modality ablations
print("--- Modality Ablations (30 epochs each) ---\n")
mod_configs = [
    ("V only",    False, False),
    ("V + T",     True,  False),
    ("V + A",     False, True),
    ("V + T + A", True,  True),
]

mod_results = []
for name, ut, ua in mod_configs:
    print(f"  {name}...", end=" ", flush=True)
    m = EFAblation(use_text=ut, use_audio=ua, clip_ids=[0,1,2]).to(DEVICE)
    tw, tu = quick_train(m, train_loader, val_loader, test_loader)
    mod_results.append((name, tw, tu))
    print(f"Test WAR={tw:.1f}%, UAR={tu:.1f}%")

print(f"\n{'Config':<15} | {'Test WAR':>8} | {'Test UAR':>8}")
print("-" * 40)
for n, tw, tu in mod_results:
    print(f"{n:<15} | {tw:>7.1f}% | {tu:>7.1f}%")

# %%
# 3b: Clip ablations
print("\n--- Clip Ablations (30 epochs each) ---\n")
clip_configs = [
    ("I only",       [0]),
    ("II only",      [1]),
    ("III only",     [2]),
    ("I + II",       [0, 1]),
    ("I + III",      [0, 2]),
    ("II + III",     [1, 2]),
    ("I + II + III", [0, 1, 2]),
]

clip_results = []
for name, cids in clip_configs:
    print(f"  {name}...", end=" ", flush=True)
    m = EFAblation(use_text=True, use_audio=True, clip_ids=cids).to(DEVICE)
    tw, tu = quick_train(m, train_loader, val_loader, test_loader)
    clip_results.append((name, tw, tu))
    print(f"Test WAR={tw:.1f}%, UAR={tu:.1f}%")

print(f"\n{'Config':<15} | {'Test WAR':>8} | {'Test UAR':>8}")
print("-" * 40)
for n, tw, tu in clip_results:
    print(f"{n:<15} | {tw:>7.1f}% | {tu:>7.1f}%")

--- Modality Ablations (30 epochs each) ---

  V only... 

/tmp/ipykernel_58/2553630346.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
/tmp/ipykernel_58/4100598148.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.inter_trans = nn.TransformerEncoder(el, num_layers=2)


Test WAR=33.8%, UAR=21.5%
  V + T... Test WAR=36.6%, UAR=24.1%
  V + A... Test WAR=36.6%, UAR=25.5%
  V + T + A... Test WAR=35.7%, UAR=25.2%

Config          | Test WAR | Test UAR
----------------------------------------
V only          |    33.8% |    21.5%
V + T           |    36.6% |    24.1%
V + A           |    36.6% |    25.5%
V + T + A       |    35.7% |    25.2%

--- Clip Ablations (30 epochs each) ---

  I only... Test WAR=34.5%, UAR=24.9%
  II only... Test WAR=34.3%, UAR=25.2%
  III only... Test WAR=31.2%, UAR=23.7%
  I + II... Test WAR=35.2%, UAR=23.3%
  I + III... Test WAR=35.7%, UAR=24.1%
  II + III... Test WAR=35.9%, UAR=21.1%
  I + II + III... Test WAR=35.2%, UAR=25.4%

Config          | Test WAR | Test UAR
----------------------------------------
I only          |    34.5% |    24.9%
II only         |    34.3% |    25.2%
III only        |    31.2% |    23.7%
I + II          |    35.2% |    23.3%
I + III         |    35.7% |    24.1%
II + III        |    35.9% |    21.1%

In [14]:
# %% [markdown]
# ---
# ## 14. Analysis 4: Transition Structure Deep Dive

# %%
# 4a: Accuracy by A→B transition
print("--- Accuracy by A→B Transition ---\n")
print(f"{'A':>10} → {'B':>10} | {'Emp%':>5} | {'n':>4} | {'Acc':>5}")
print("-" * 50)
for a in range(7):
    for b in range(7):
        m = (meta['clip3_emotion'] == a) & (labels == b)
        n = m.sum()
        if n < 2: continue
        acc = (preds[m] == b).sum() / n * 100
        emp = train_trans_prob[a][b] * 100
        print(f"{emo_names[a]:>10} → {emo_names[b]:>10} | {emp:>4.1f}% | {n:>4} | {acc:>4.1f}%")

# %%
# 4b: Does A's emotion change the prediction distribution?
print("\n--- Model's Prediction Distribution by A's Emotion ---\n")
print(f"{'A emotion':>10} | Top predicted B classes")
print("-" * 60)
for a in range(7):
    m = meta['clip3_emotion'] == a
    if m.sum() < 5: continue
    dist = Counter(preds[m])
    total = m.sum()
    top = ", ".join(f"{emo_names[c]}:{100*cnt/total:.0f}%" for c, cnt in dist.most_common(3))
    print(f"{emo_names[a]:>10} (n={total:>3}) | {top}")

print(f"\n{'A emotion':>10} | Expected B (empirical)")
print("-" * 60)
for a in range(7):
    if train_trans[a].sum() < 5: continue
    top = np.argsort(-train_trans_prob[a])[:3]
    top_str = ", ".join(f"{emo_names[c]}:{train_trans_prob[a][c]*100:.0f}%" for c in top)
    print(f"{emo_names[a]:>10}         | {top_str}")

# %%
# 4c: Model confidence on likely vs unlikely transitions
likely, unlikely = [], []
for i in range(len(labels)):
    a = meta['clip3_emotion'][i]
    b = labels[i]
    if a < 0 or a >= 7: continue
    tp = train_trans_prob[a][b]
    conf = probs[i, b]
    if tp > 0.2: likely.append(conf)
    elif tp < 0.05: unlikely.append(conf)

print(f"\n--- Confidence: Likely vs Unlikely Transitions ---")
print(f"  Likely (emp>20%):   avg P(true) = {np.mean(likely)*100:.1f}% (n={len(likely)})")
print(f"  Unlikely (emp<5%):  avg P(true) = {np.mean(unlikely)*100:.1f}% (n={len(unlikely)})")
if np.mean(unlikely) > 0:
    print(f"  Ratio: {np.mean(likely)/np.mean(unlikely):.2f}x")

--- Accuracy by A→B Transition ---

         A →          B |  Emp% |    n |   Acc
--------------------------------------------------
     angry →      angry | 34.5% |   29 | 44.8%
     angry →    disgust | 12.5% |    8 | 12.5%
     angry →       fear |  2.4% |    2 |  0.0%
     angry →      happy | 12.3% |   15 | 13.3%
     angry →    neutral | 11.8% |   17 | 41.2%
     angry →        sad | 17.0% |   17 | 29.4%
     angry →   surprise |  9.5% |    3 |  0.0%
   disgust →      angry | 27.8% |   13 | 38.5%
   disgust →    disgust | 15.4% |    9 | 44.4%
   disgust →      happy | 22.8% |    7 | 71.4%
   disgust →    neutral | 16.7% |    8 | 50.0%
   disgust →        sad |  8.0% |    2 |  0.0%
      fear →      angry | 24.0% |    2 | 50.0%
      fear →      happy | 16.0% |    2 |  0.0%
      fear →    neutral | 12.0% |    2 | 50.0%
     happy →      angry | 12.0% |    7 | 42.9%
     happy →    disgust |  7.3% |    8 | 25.0%
     happy →      happy | 46.3% |   45 | 53.3%
     happy →    neut